In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.aml_silver.stg_transactions")
df.show(5)

In [0]:
df = df.withColumn(
    "is_same_account",
    F.when(F.col("from_account") == F.col("to_account"), 1).otherwise(0)
)

df.select("from_account", "to_account", "is_same_account").show(10)

In [0]:
df = df.withColumn(
    "is_currency_mismatch",
    F.when(F.col("payment_currency") != F.col("receiving_currency"), 1).otherwise(0)
)

df.select("payment_currency", "receiving_currency", "is_currency_mismatch").show(10)

In [0]:
df = df.withColumn(
    "amount_discrepancy",
    F.abs(F.col("amount_paid") - F.col("amount_received")),
)
df.select("amount_paid", "amount_received", "amount_discrepancy").show(10)



In [0]:
from pyspark.sql import Window

window_24h = (
    Window
    .partitionBy("to_account")
    .orderBy(F.col("transaction_timestamp").cast("long"))
    .rangeBetween(-86400, 0)
)

df = df.withColumn(
    "unique_senders_24h",
    F.size(F.collect_set("from_account").over(window_24h))
)
df.select("to_account", "transaction_timestamp", "unique_senders_24h").show(10)

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.aml_silver.pyspark_transaction_features")
)

print(f"Wrote {df.count():,} rows to workspace.aml_silver.pyspark_transaction_features")